# Phase 4 — Feature Engineering

This notebook evaluates six candidate features without using the final test set for model selection. We first reserve a fixed 15% test holdout, then perform five-fold cross-validation only on the remaining development rows.

The model used here is a lightweight **diagnostic estimator**, not the final price model. Phase 4 does not save a trained model or report test performance.

## 1. Imports and project paths

In [ ]:
from pathlib import Path
import sys

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    FEATURE_REPORT_PATH, FEATURE_SUMMARY_PATH, HOLDOUT_ASSIGNMENT_PATH,
    PROCESSED_DATA_PATH, TABLES_DIR,
)
from src.features import (
    CANDIDATE_FEATURES, CV_RESULTS_FILENAME, FOLD_RESULTS_FILENAME,
    LUXURY_BRANDS, create_holdout_assignment, engineer_features,
    load_cleaned_features, run_feature_experiment,
)
from src.validate_data import file_sha256

print(f"Project root: {PROJECT_ROOT}")
print(f"Cleaned input: {PROCESSED_DATA_PATH}")

## 2. Load and identify the cleaned input

The source hash will be checked again at the end. Feature engineering returns new DataFrames and never overwrites the cleaned CSV.

In [ ]:
cleaned_hash_before = file_sha256(PROCESSED_DATA_PATH)
cars = load_cleaned_features(PROCESSED_DATA_PATH)

print(f"Shape: {cars.shape}")
print(f"Cleaned SHA-256: {cleaned_hash_before}")
print(f"Missing seats still unimputed: {int(cars['seats'].isna().sum())}")
display(cars.head())

## 3. Understand the six candidate features

These calculations are deterministic and row-level: they do not learn from the target or from other rows. Undefined ratios remain missing so the training pipeline can impute them inside each fold.

The luxury-brand flag uses an explicit documented mapping. It is a heuristic, not a claim that every buyer or market uses the same definition.

In [ ]:
engineered_preview = engineer_features(cars.head(10))
preview_columns = [
    'brand', 'vehicle_age', 'km_driven', 'engine', 'max_power', 'seats',
    *CANDIDATE_FEATURES,
]
display(engineered_preview[preview_columns])
print(f"Documented luxury brands ({len(LUXURY_BRANDS)}):")
print(', '.join(sorted(LUXURY_BRANDS)))

## 4. Reserve the untouched test holdout

Price-band stratification keeps inexpensive, mid-range, and luxury listings represented in both partitions. Phase 4 records only the assignment and counts. It does not calculate test MAE, RMSE, R², or inspect test predictions.

In [ ]:
holdout_assignment = create_holdout_assignment(cars)
split_counts = holdout_assignment['split'].value_counts().rename('rows').to_frame()
band_counts = pd.crosstab(
    holdout_assignment['price_band'],
    holdout_assignment['split'],
)

display(split_counts)
display(band_counts)
print("The test rows are now reserved and will not enter feature cross-validation.")

## 5. Run development-only feature evaluation

For each experiment, preprocessing is fitted separately inside each training fold. The diagnostic estimator trains on `log1p(selling_price)`, then predictions are converted back to rupees before MAE and RMSE are calculated.

This cell normally takes under a minute on the tested environment.

In [ ]:
feature_summary = run_feature_experiment()
print("Phase 4 experiment complete.")
print(f"Development rows: {feature_summary['holdout']['development_rows']:,}")
print(f"Reserved test rows: {feature_summary['holdout']['test_rows']:,}")
print(f"Test set evaluated: {feature_summary['test_set_evaluated']}")

## 6. Compare the evidence

A positive improvement means lower MAE than the raw-feature baseline. Promotion is intentionally conservative: at least 1% mean MAE improvement and at least four of five better folds. Tiny differences can easily be cross-validation noise.

In [ ]:
cv_results = pd.read_csv(TABLES_DIR / CV_RESULTS_FILENAME)
result_columns = [
    'experiment', 'added_features', 'mean_mae_inr',
    'mae_improvement_inr', 'mae_improvement_percentage',
    'folds_better_than_baseline', 'recommended_for_phase5',
]
display(cv_results[result_columns].sort_values('mean_mae_inr'))

recommended = cv_results.loc[cv_results['recommended_for_phase5'], 'added_features'].tolist()
print(f"Features promoted as Phase 5 defaults: {recommended or 'None'}")
print("The best observed reduction is only about 0.33%, so no feature clears the 1% rule.")

## 7. Feature comparison chart

Error bars show fold-to-fold variation in the paired MAE improvement. Every interval crosses zero, which reinforces why the observed differences should not be treated as dependable gains.

![Feature engineering MAE comparison](../reports/figures/11_feature_engineering_mae_comparison.png)

## 8. Final verification and conclusion

The raw columns remain the Phase 5 default. Candidate formulas remain available for model-specific experiments later, but none is accepted merely because it sounds useful.

In [ ]:
cleaned_hash_after = file_sha256(PROCESSED_DATA_PATH)
fold_results = pd.read_csv(TABLES_DIR / FOLD_RESULTS_FILENAME)

assert cleaned_hash_after == cleaned_hash_before
assert feature_summary['holdout']['development_rows'] == 12_957
assert feature_summary['holdout']['test_rows'] == 2_287
assert feature_summary['test_set_evaluated'] is False
assert feature_summary['production_model_trained_or_saved'] is False
assert feature_summary['recommended_experiments_for_phase5'] == []
assert len(cv_results) == 8
assert len(fold_results) == 40

print("Phase 4 verification passed.")
print(f"Detailed report: {FEATURE_REPORT_PATH}")
print("Next: Phase 5 will create the 70/15/15 split and train the dummy median baseline.")